# Calling LLMs: A Step-by-Step Approach

In this Jupyter notebook, we will dive deep into the world of LLMs and demonstrate how to use them effectively. First, we will explore how to deploy an LLM, and then we will see how to connect to it.


## Step 1: Install the Necessary Libraries

In [1]:
!pip install requests

In [2]:
!pip install openai

In [3]:
import requests
import subprocess
import json
from openai import OpenAI

## Step 2: Configure the Model Pod (only appliable for Model Manager deployments)

NOTE: unless you are using [Method C: using Model Manager](deployment_with_model_manager.md) explained in the README for Model Deployment, you can skip this step.

In [4]:
!kubectl get pods

NAME                                                    READY   STATUS    RESTARTS   AGE
datallm-0                                               2/2     Running   0          8d
fs-76d5f96db9-fx4zs                                     2/2     Running   0          30d
llama3-8b-predictor-00001-deployment-5596554cc4-24sgd   1/3     Running   0          13s
ml-pipeline-ui-artifact-567b5dc4b7-pghj5                2/2     Running   0          30d
ml-pipeline-visualizationserver-857b954464-pgngm        2/2     Running   0          30d


Now, ensure you have the 'access token'. You should see in the lables the line **hpe-ezua/add-auth-token=true**. See the documentation for more information. 

## Step 3: Obtaining the Auth Token

The secret mounts the notebook's filesystem in the local pod at **/etc/secrets/ezua/.auth_token**. The notebook reads the token from the file and puts the token in an outgoing request. The notebook can do this repeatedly because the token service in Unified Analytics regularly refreshes the access token.

In [6]:
secret_file_path = "/etc/secrets/ezua/.auth_token"

with open(secret_file_path, "r") as file:
    token = file.read().strip()

print(f"Authorization Token: {token}")

Authorization Token: eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJVTF80eW9jY19fSXBJanZlQ0NYN1lqcnl5eFpUV09GQlVQWEhyeF9wM05NIn0.eyJleHAiOjE3MzA5MDc5MjcsImlhdCI6MTczMDkwNjEyNywiYXV0aF90aW1lIjoxNzMwOTA2MTI2LCJqdGkiOiI4ZDBlMWRhOC05MjNkLTRjN2MtODNlMS1mNDIyZjU5MzE0MmIiLCJpc3MiOiJodHRwczovL2tleWNsb2FrLnByb2QuZGlzY292ZXIuaHBlcGNhaS5jb20vcmVhbG1zL1VBIiwic3ViIjoiMzc4NDlhZDItNWQyNS00MjViLWE0N2UtMzA4Zjk3ZDAwYWQ5IiwidHlwIjoiQmVhcmVyIiwiYXpwIjoidWEiLCJub25jZSI6Im9USlR2T082UmJVZmItVDEzYWF2SG9Bb0hBZjNyb2VZbkcyOEpxY0p3aEEiLCJzZXNzaW9uX3N0YXRlIjoiYTZjNmQ1ZTktODJlMi00ZWU4LWI0MWEtMTkzZjM4NDQ0OWY1IiwiYWNyIjoiMSIsInNjb3BlIjoib3BlbmlkIGVtYWlsIHJhY2syLW1pbDp1YSBwcm9maWxlIG9mZmxpbmVfYWNjZXNzIiwic2lkIjoiYTZjNmQ1ZTktODJlMi00ZWU4LWI0MWEtMTkzZjM4NDQ0OWY1IiwidWlkIjoiMTAwMDAwNzYiLCJlbWFpbF92ZXJpZmllZCI6ZmFsc2UsImdpZCI6IjEwMDEiLCJuYW1lIjoiQWxiYSBTYW5jaGV6IiwibmFtZXNwYWNlIjoiYWxiYS1zYW5jaGV6MS1ocGUtY29tLTVmOWI5ZTQzIiwiZ3JvdXBzIjpbInVhLWVuYWJsZWQiLCJvZmZsaW5lX2FjY2VzcyIsImFkbWluIiwidW1hX2F1dGhvcml6YXRpb24iLCJkZWZ


### NOTE: 
*This token has a short lifespan (10 minutes). If you encounter the error 'JWT expired,' please run this step and the curl command again.*


## Step 4: Model Interaction

Retrieve the URL of the inference service you are going to use, e.g.:

In [ ]:
!kubectl get isvc

NOTE: 
If you are using deployment_with_model_manager file, you can find it in the section **Tools & Frameworks > Data Engineering > Model Manager > Manage > InferenceService > URL**.    
For more information, please refer to the documentation.

In fhe following cell, replace the value of the `url` variable with the desired inference URL found above:

In [26]:
url = "https://llama3-8b-predictor-alba-sanchez1-hpe-com-5f9b9e43.prod.discover.hpepcai.com"
endpoint = f'{url}/v1/completions'

### Example 1: Using the curl command. 

Curl is a command-line tool used to trasnfer data to or from server using various protocols, including HTTP. We can interacting with web service. To use curl, you need to include specific headers:

- accept: application/json: You expect a response in JSON format.
- Authorization: Bearer {your_token}: Your actual token.
- Content-Type: application/json: The type of data you are sendin to the server,which in this case is JSON. 
- Configuration of the instruction: This involves specifying the command option: for example we are using the model llama3-8b-instruction.


In [27]:
curl_command = [
    'curl',
    '-H','accept: application/json',
    '-H', f'Authorization: Bearer {token}', 
    '-H', 'Content-Type: application/json',
    '-d', '{"model": "meta/llama3-8b-instruct", "prompt": "Once upon a time", "max_tokens": 64}',
    endpoint
]

result = subprocess.run(curl_command, capture_output=True, text=True)

In [29]:
# Assuming result.stdout contains the server's JSON response
response = json.loads(result.stdout)

# Check if the response was successful
if 'choices' in response and len(response['choices']) > 0:
    print("Response successful.")
    # Extract and display the generated text
    print("-----Generated text:---------------")
    generated_text = response['choices'][0]['text']
    print(generated_text.strip())
else:
    print("Response not successful.")
    # You can display more information if necessary
    if 'detail' in response:
        print("Error details:", response['detail'])
    else:
        print("Server response:", response)

Response successful.
-----Generated text:---------------
, in a small village nestled in the rolling hills of the countryside, there lived a young girl named Sophie. Sophie was a curious and adventurous child, with a mop of curly brown hair and a smile that could light up a room. She loved to explore the world around her, and was constantly asking questions about the mysteries


## Example 2: Basic Text Generation. 

We have our API set up so let's explore basic text generation with our LLM. Basic Text Generation in a Large Language Model (LLM) refers to the capability of these models to produce coherent and contextually relevant text based on the input they receive.

In [10]:
headers = {
    "accept": "application/json",
    "Content-Type": "application/json",
    "Authorization": f"Bearer {token}"
}

In [30]:
def call_api(prompt, api_key=token):
    try:
        # Initialize the OpenAI client
        client = OpenAI(
            base_url=endpoint,
            api_key=api_key
        )

        # Create the completion request
        completion = client.chat.completions.create(
            model= "meta/llama3-8b-instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5,
            top_p=0.7,
            max_tokens=500,
            stream=True
        )

        # Collect and return the full response
        response_text = ""
        for chunk in completion:
            if chunk.choices[0].delta.content is not None:
                response_text += chunk.choices[0].delta.content

        return response_text
    except Exception as e:
        print(f"An exception occurred: {e}")
        print(f"Looks like the User has the wrong access Token")
        if str(e) == "RBAC: access denied":
            return f"Looks like your user has the wrong access token and isn't privileged to access the model"
        else:
            return f"The fronted couldn't establish a connection to the model at the following URL: {endpoint}."

In [31]:
def generate_text(prompt, max_tokens=100):
    """Generate text using the LLM API.
    
    Args:
        prompt (str): The input prompt for the LLM.
        max_tokens (int): Maximum number of tokens to generate.
    
    Returns:
        str: The generated text or an error message.
    """
    data = {
        "model": "meta/llama3-8b-instruct",
        "prompt": prompt,
        "max_tokens": max_tokens
    }
    
    response = requests.post(endpoint, headers=headers, json=data)
    
    if response.status_code == 200:
        return response.json()["choices"][0]["text"].strip()
    else:
        return f"Error: {response.status_code} - {response.text}"

In [32]:
# Test the API connection
test_prompt = "Hello world!"
print("---------------------------")
print(generate_text(test_prompt))

---------------------------
Today, I'm excited to share with you my updated Top 10 Must-Have Fashion Items Every Woman Should Own in Her Wardrobe.
After some careful consideration and research, I've compiled a list of essential fashion items that can be mixed and matched to create a versatile and stylish wardrobe. You can wear these pieces alone or with other items to create multiple outfits.

Before we dive in, I want to clarify that this list is tailored to women who want to create a solid foundation for their wardrobe without


## Example 3: Text Summarization

Text summarization is the task of creating a concise and coherent summary of a longer text while preserving its key information. LLMs can be quite effective at this task when given appropriate prompts. Let's try summarizing a paragraph about the Industrial Revolution:

In [33]:
summarization_prompt = """
Summarize the following paragraph in one or two sentences:

The Renaissance, which flourished from the 14th to the 17th centuries, was a period marked by a revival of interest in art, science, and classical antiquity in Europe. Originating in Italy, this cultural movement led to significant developments in painting, literature, and humanism, fostering creativity and innovation that laid the groundwork for the modern world.

Summary:
"""

print("Text Summarization result:")
print("---------------------------")
print(generate_text(summarization_prompt, max_tokens=50))

Text Summarization result:
---------------------------
The Renaissance was a cultural movement in Europe from the 14th to 17th centuries that brought about a revival in art, science, and classical antiquity, leading to significant developments in various fields, including painting, literature, and humanism.


## Example 4: Question Answering

Question Answeting is the task of to understand and respond to questions posed by users. This involves analyzing the input question, retrieving relevant information from their training data, and generating an accurate and coherent response.

In [34]:
qa_prompt = """
Answer the following question based on the given context:

Context: A small town in the countryside is preparing for its annual harvest festival. The locals are excited about the event, which features a parade, food stalls with local produce, and various competitions like pumpkin carving and pie-eating contests. Everyone is involved in the preparations, from decorating the town square to baking pies and organizing games for children.

Question: What are some traditional activities that townspeople might include in a harvest festival, and how do these activities reflect the community's culture?

Answer:
"""

print("Question Answering result:")
print("---------------------------")
print(generate_text(qa_prompt, max_tokens=50))

Question Answering result:
---------------------------
In a harvest festival, townspeople might include traditional activities such as:

1. Pumpkin carving: Pumpkins are a ubiquitous symbol of fall harvests in many cultures. Carving intricate designs and patterns on pumpkins is a long-standing tradition that not only


# Example 5: Text simplification

Text simplification is the process of transforming complex or dense text into clearer, more accessible language.

In [35]:
simplification_prompt = """
Simplify the following paragraph by using simpler language and making it easier to understand: .. 
Global warming is a long-term increase in Earth's average surface temperature due to human activities, primarily the burning of fossil fuels, which increases the levels of greenhouse gases in the atmosphere. This phenomenon has far-reaching effects on climate patterns, leading to more severe weather events, rising sea levels, and disruptions to ecosystems.
Simplified Text:

"""
print("Text simplification result:")
print("---------------------------")
print(generate_text(simplification_prompt, max_tokens=50))

Text simplification result:
---------------------------
Global warming is when the Earth gets warmer over time. This is mainly because of people burning fossil fuels, like coal, oil, and gas. These fuels release bad gases that trap heat in the air, making the Earth hotter. This makes extreme weather
